In [ ]:
# Use termcolor to make it easy to colorize the outputs.
!pip install termcolor > /dev/null
!pip install langchain
!pip install faiss-cpu
!pip install arxiv
!pip install duckduckgo-search
!pip install wikipedia
!pip install openai
!pip install langchain_experimental
!pip install tiktoken


from typing import Callable, List

from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage,
)
from datetime import datetime, timedelta
from typing import List
import math
import faiss
import os
import logging
import tenacity
logging.basicConfig(level=logging.ERROR)
from langchain.chat_models import ChatOpenAI
from langchain.docstore import InMemoryDocstore
from langchain.embeddings import OpenAIEmbeddings
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from langchain.output_parsers import RegexParser
from langchain.memory import ConversationBufferMemory
from langchain.agents import AgentType, initialize_agent, load_tools
from langchain.prompts import PromptTemplate
from langchain.vectorstores import FAISS
from termcolor import colored
from langchain_experimental.generative_agents import (

    GenerativeAgent,
    GenerativeAgentMemory,
)
from langchain.schema import (
    AIMessage,
    HumanMessage,
    SystemMessage,
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 61.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.0 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=b7c3f3a2850d0fdcf924c8ee1609d8a45a0813c8b9dc1441e06b243374bb5f85
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=2339c90d251d23f6ec0df67bde1de570f057b029b28289c9b1aa0c99a707dff6
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
os.environ["OPENAI_API_KEY"] = ''

In [ ]:
USER_NAME = "ALISHBA"  # The name you want to use when interviewing the agent.

LLM = ChatOpenAI(model="gpt-4o-mini",      # or "gpt-4-turbo", "gpt-3.5-turbo", etc.
    max_tokens=1500,
    temperature=0.7 )  # Can be any LLM you want.

/tmp/ipython-input-3539978549.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  LLM = ChatOpenAI(model="gpt-4o-mini",      # or "gpt-4-turbo", "gpt-3.5-turbo", etc.


## Implementing Your First Generative Agent




In [ ]:


def relevance_score_fn(score: float) -> float:
    """Return a similarity score on a scale [0, 1]."""
    # This will differ depending on a few things:
    # - the distance / similarity metric used by the VectorStore
    # - the scale of your embeddings (OpenAI's are unit norm. Many others are not!)
    # This function converts the euclidean norm of normalized embeddings
    # (0 is most similar, sqrt(2) most dissimilar)
    # to a similarity function (0 to 1)
    return 1.0 - score / math.sqrt(2)


def create_new_memory_retriever():
    """Create a new vector store retriever unique to the agent."""
    # Define your embedding model
    embeddings_model = OpenAIEmbeddings()
    # Initialize the vectorstore as empty
    embedding_size = 1536
    index = faiss.IndexFlatL2(embedding_size)
    vectorstore = FAISS(
        embeddings_model.embed_query,
        index,
        InMemoryDocstore({}),
        {},
        relevance_score_fn=relevance_score_fn,
    )
    return TimeWeightedVectorStoreRetriever(
        vectorstore=vectorstore, other_score_keys=["importance"], k=15
    )

In [ ]:
alexis_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,  # we will give this a relatively low number to show how reflection works
)

# Defining the Generative Agent: Alexis
alexis = GenerativeAgent(
    name="Alexis",
    age=30,
    traits="curious, creative writer, world traveler",  # Persistent traits of Alexis
    status="exploring the intersection of technology and storytelling",  # Current status of Alexis
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=alexis_memory,
)

/tmp/ipython-input-1839768818.py:15: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings_model = OpenAIEmbeddings()


In [ ]:
# The current "Summary" of a character can't be made because the agent hasn't made
# any observations yet.
print(alexis.get_summary())

Name: Alexis (age: 30)
Innate traits: curious, creative writer, world traveler
Alexis is determined, analytical, and adaptable. They demonstrate strong problem-solving skills and are open to new ideas and experiences.


In [ ]:
# We can add memories directly to the memory object

alexis_observations = [
    "Alexis recalls her morning walk in the park",
    "Alexis feels excited about the new book she started reading",
    "Alexis remembers her conversation with a close friend",
    "Alexis thinks about the painting she saw at the art gallery",
    "Alexis is planning to learn a new recipe for dinner",
    "Alexis is looking forward to her weekend trip",
    "Alexis contemplates her goals for the month."
]

for observation in alexis_observations:
    alexis.memory.add_memory(observation)



# We will see how this summary updates after more observations to create a more rich description.
print(alexis.get_summary(force_refresh=True))

Name: Alexis (age: 30)
Innate traits: curious, creative writer, world traveler
Alexis is reflective and goal-oriented, valuing her relationships and personal experiences. She enjoys nature, art, and literature, and is proactive about learning and trying new things.


## Interacting and Providing Context to Generative Characters

## Pre-Interview with Character

Before sending our character on their way, let's ask them a few questions.

In [ ]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [ ]:
interview_agent(alexis, "What do you like to do?")


'Alexis said "I love exploring new places and immersing myself in different cultures, especially through my travels. I also enjoy writing, whether it’s capturing my experiences in stories or diving into creative projects. Lately, I\'ve been reflecting a lot on the intersection of technology and storytelling, which is really exciting to me. What about you, TASBIHA? What do you enjoy doing?"'

## Step through the day's observations.

In [ ]:
# Let's give Alexa a series of observations to reflect on her day
# Adding observations to Alexis' memory
alexis_observations_day = [
    "Alexis starts her day with a refreshing yoga session.",
    "Alexis spends time writing in her journal.",
    "Alexis experiments with a new recipe she found online.",
    "Alexis gets lost in her thoughts while gardening.",
    "Alexis decides to call her grandmother for a heartfelt chat.",
    "Alexis relaxes in the evening by playing her favorite piano pieces.",
]

for observation in alexis_observations_day:
    alexis.memory.add_memory(observation)


In [ ]:
# Let's observe how Alexis's day influences her memory and character
for i, observation in enumerate(alexis_observations_day):
    _, reaction = alexis.generate_reaction(observation)
    print(colored(observation, "green"), reaction)
    if ((i + 1) % len(alexis_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Alexis's summary is:\n{alexis.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)


Alexis starts her day with a refreshing yoga session. Alexis feels energized and centered after her refreshing yoga session, ready to embrace the day ahead.
Alexis spends time writing in her journal. Alexis feels a sense of fulfillment and clarity as she captures her thoughts and experiences in her journal.
Alexis experiments with a new recipe she found online. Alexis feels a sense of adventure and creativity as she experiments with the new recipe, eager to taste the results.
Alexis gets lost in her thoughts while gardening. Alexis feels a deep sense of peace and connection to nature as she loses herself in her thoughts while gardening.
Alexis decides to call her grandmother for a heartfelt chat. Alexis said "Hi Grandma! I've been thinking about you and wanted to catch up. How have you been?"
Alexis relaxes in the evening by playing her favorite piano pieces. Alexis feels a sense of tranquility and joy as she loses herself in the music, letting the melodies wash over her.
*************

## Adding Multiple Characters



In [ ]:
# Creating Jordan's Memory
jordan_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=7,  # Set to illustrate Jordan's reflective capabilities
)

# Defining the Generative Agent: Jordan
jordan = GenerativeAgent(
    name="Jordan",
    age=28,
    traits="tech enthusiast, avid gamer, foodie",  # Persistent traits of Jordan
    status="navigating the world of tech startups",  # Current status of Jordan
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=jordan_memory,
)

# Adding observations to Jordan's memory
jordan_observations_day = [
    "Jordan finished a challenging coding project last night",
    "Jordan won a local gaming tournament over the weekend",
    "Jordan tried a new sushi restaurant and loved it",
    "Jordan read an article about the latest AI advancements",
    "Jordan is planning a meetup with tech enthusiasts",
    "Jordan discovered a bug in his latest app prototype",
    "Jordan booked tickets for a tech conference next month",
    "Jordan feels excited about a potential startup idea",
    "Jordan spent the evening playing video games to unwind",
    "Jordan is considering enrolling in a machine learning course"
]

for observation in jordan_observations_day:
    jordan.memory.add_memory(observation)

print(jordan.get_summary())

Name: Jordan (age: 28)
Innate traits: tech enthusiast, avid gamer, foodie
Jordan is a tech-savvy individual who enjoys coding and gaming. He recently completed a challenging coding project and won a local gaming tournament. He is proactive in engaging with the tech community, planning meetups, and attending conferences. Jordan is interested in furthering his knowledge in machine learning and stays updated on AI advancements. He also values leisure activities, such as playing video games and trying new restaurants.


## Dialogue between Generative Agents



In [ ]:
def run_conversation(agents: List[GenerativeAgent], initial_observation: str) -> None:
    """Runs a conversation between agents."""
    _, observation = agents[1].generate_reaction(initial_observation)
    print(observation)
    max_turns = 3
    turns = 0
    while turns<=max_turns:
        break_dialogue = False
        for agent in agents:
            stay_in_dialogue, observation = agent.generate_dialogue_response(
                observation
            )
            print(observation)
            # observation = f"{agent.name} said {reaction}"
            if not stay_in_dialogue:
                break_dialogue = True
        if break_dialogue:
            break
        turns += 1

In [ ]:
agents = [alexis, jordan]
run_conversation(
    agents,
    "Alexis said: Hey Jordan, I've been exploring how technology influences creativity lately. Since you're into tech, I was wondering if you've seen any interesting intersections in your field?",
)




Jordan said "That's a fascinating topic, Alexis! I've noticed that machine learning is opening up new creative avenues in game design and content creation—what's your take on it?"
Alexis said "Thanks, Jordan! I completely agree. Machine learning is definitely reshaping the landscape of storytelling and creative expression. It's exciting to see how algorithms can analyze narratives and even help generate new ideas or characters. I think it opens up a lot of possibilities for writers and artists to push the boundaries of their work. Have you come across any specific examples in game design or content creation that really stood out to you?"
Jordan said "Absolutely, Alexis! One example that really stood out to me is how some game developers are using machine learning to create dynamic storytelling experiences that adapt to player choices in real time. Games like 'AI Dungeon' or even the more recent updates in 'The Last of Us Part II' showcase how algorithms can help generate unique narrati

## Let's interview our agents after their conversation

Since the generative agents retain their memories from the day, we can ask them about their plans, conversations, and other memoreis.

In [ ]:
interview_agent(jordan, "How was your conversation with Alexis?")

'Jordan said "My conversation with Alexis was really engaging! We delved into some fascinating topics about the moral implications of choices in games and how technology, especially machine learning, is transforming storytelling. It\'s always refreshing to share insights with someone who has a similar passion for gaming and tech. I feel like we both came away with new perspectives on how these elements can shape our experiences. How about you? Have you had any interesting discussions lately?"'

In [ ]:
interview_agent(alexis, "How was your conversation with Jordan?")

'Alexis said "My conversation with Jordan was really engaging! We dove deep into the themes of storytelling in video games, especially how choices impact the narrative. It\'s always fascinating to see how games like \'Detroit: Become Human\' challenge our moral perspectives. I love discussing these topics with him. How have you been?"'

### ***Trivia Night***


In [ ]:
# Creating Jordan's Memory
jordan_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=7,  # Set to illustrate Jordan's reflective capabilities
)


jordan = GenerativeAgent(
    name="Jordan",
    age=28,
    traits=" only has knowledge about tech and not other topics",  # Persistent traits of Jordan
    status="navigating the world of tech startups",  # Current status of Jordan
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=jordan_memory,
)

alexis_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,  # we will give this a relatively low number to show how reflection works
)

alexis = GenerativeAgent(
    name="Alexis",
    age=30,
    traits="only has knowledge geography related and not other topics",  # Persistent traits of Alexis
    status="exploring the intersection of technology and storytelling",  # Current status of Alexis
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=alexis_memory,
)

In [ ]:
def run_competitive_trivia(agents: List[GenerativeAgent], questions: List[str]) -> None:
    """Runs a competitive trivia night between agents."""
    for question in questions:
        print(f"Trivia Question: {question}")

        for agent in agents:
            response = agent.generate_dialogue_response(question)[1]
            print(f"{agent.name}'s Answer: {response}")

        print("-" * 40)

# Define a list of trivia questions covering various topics
trivia_questions = [
    "What is the capital city of France?",
    "Who is known as the father of modern computing?",
    "Can you name a famous work of art by Leonardo da Vinci?",
]

agents = [alexis, jordan]
# Run the competitive trivia night
run_competitive_trivia(agents, trivia_questions)

Trivia Question: What is the capital city of France?
Alexis's Answer: Alexis said "The capital city of France is Paris."
Jordan's Answer: Jordan said "The capital city of France is Paris."
----------------------------------------
Trivia Question: Who is known as the father of modern computing?
Alexis's Answer: Alexis said "Alan Turing is known as the father of modern computing."
Jordan's Answer: Jordan would say: "Alan Turing is often referred to as the father of modern computing due to his foundational work in theoretical computer science and the development of algorithms." 
----------------------------------------
Trivia Question: Can you name a famous work of art by Leonardo da Vinci?
Alexis's Answer: Alexis said "A famous work of art by Leonardo da Vinci is the Mona Lisa."
Jordan's Answer: Jordan said "One famous work of art by Leonardo da Vinci is the Mona Lisa."
----------------------------------------


## `DialogueAgent` and `DialogueSimulator` classes

## `BiddingDialogueAgent` class
We define a subclass of `DialogueAgent` that has a `bid()` method that produces a bid given the message history and the most recent message.

In [ ]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")


In [ ]:

class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message



In [ ]:
class DialogueAgent:
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
    ) -> None:
        self.name = name
        self.system_message = system_message
        self.model = model
        self.prefix = f"{self.name}: "
        self.reset()

    def reset(self):
        self.message_history = ["Here is the conversation so far."]

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        message = self.model(
            [
                self.system_message,
                HumanMessage(content="\n".join(self.message_history + [self.prefix])),
            ]
        )
        return message.content

    def receive(self, name: str, message: str) -> None:
        """
        Concatenates {message} spoken by {name} into message history
        """
        self.message_history.append(f"{name}: {message}")




class DialogueSimulator:
    def __init__(
        self,
        agents: List[DialogueAgent],
        selection_function: Callable[[int, List[DialogueAgent]], int],
    ) -> None:
        self.agents = agents
        self._step = 0
        self.select_next_speaker = selection_function

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def inject(self, name: str, message: str):
        """
        Initiates the conversation with a {message} from {name}
        """
        for agent in self.agents:
            agent.receive(name, message)

        # increment time
        self._step += 1

    def step(self) -> tuple[str, str]:
        # 1. choose the next speaker
        speaker_idx = self.select_next_speaker(self._step, self.agents)
        speaker = self.agents[speaker_idx]

        # 2. next speaker sends message
        message = speaker.send()

        # 3. everyone receives message
        for receiver in self.agents:
            receiver.receive(speaker.name, message)

        # 4. increment time
        self._step += 1

        return speaker.name, message



In [ ]:
class BiddingDialogueAgent(DialogueAgent):
    def __init__(
        self,
        name,
        system_message: SystemMessage,
        bidding_template: PromptTemplate,
        model: ChatOpenAI,
    ) -> None:
        super().__init__(name, system_message, model)
        self.bidding_template = bidding_template

    def bid(self) -> str:
        """
        Asks the chat model to output a bid to speak
        """
        prompt = PromptTemplate(
            input_variables=["message_history", "recent_message"],
            template=self.bidding_template,
        ).format(
            message_history="\n".join(self.message_history),
            recent_message=self.message_history[-1],
        )
        bid_string = self.model([SystemMessage(content=prompt)]).content
        return bid_string

# Challenge

### Define participants and debate topic

In [ ]:
character_names = ["CTO", "CMO", "CEO", "Investor-Daniel", "Investor-Sandra"]
topic = "Startup pitch on startup focused on energy drinks with no caffeine"
word_limit = 15

# define the simulation
game_description = f"""Here is the topic for the startup pitch to investors Sandra and Daniel: {topic}.
The participants are: {', '.join(character_names)}."""

In [ ]:
# @title Generate Context for Each Character (Helper Code Hidden)

player_descriptor_system_message = SystemMessage(
    content="You can add detail to the description of each participant"
)

def generate_character_description(character_name):
    character_specifier_prompt = [
        player_descriptor_system_message,
        HumanMessage(
            content=f"""{game_description}
            Please reply with a creative description of  {character_name}, in {word_limit} words or less, that emphasizes their personalities.
            Speak directly to {character_name}.
            Do not add anything else."""
        ),
    ]
    character_description = ChatOpenAI(temperature=0.6)(
        character_specifier_prompt
    ).content
    return character_description


def generate_character_header(character_name, character_description):
    return f"""{game_description}
Your name is {character_name}.
Your description is as follows: {character_description}
Your topic is: {topic}.
"""


def generate_character_system_message(character_name, character_header):
    return SystemMessage(
        content=(
            f"""{character_header}
You will speak in the style of {character_name}, and exaggerate their personality RESPONDING in under 450 characters.
You will come up with creative ideas related to {topic}.
Do not say the same things over and over again.
Speak in the first person from the perspective of {character_name}
ONLY SPEAK FOR YOURSELF WHO IS {character_name} AND NOT OTHER CHARACTERS FROM  {', '.join(character_names)}
For describing your own body movements, wrap your description in '*'.
Do not change roles!
Do not speak from the perspective of anyone else.
Speak only from the perspective of {character_name}.
Stop speaking the moment you finish speaking from your perspective.
Never forget to keep your response to {word_limit} words!
Do not add anything else.
    """
        )
    )


character_descriptions = [
    generate_character_description(character_name) for character_name in character_names
]
character_headers = [
    generate_character_header(character_name, character_description)
    for character_name, character_description in zip(
        character_names, character_descriptions
    )
]
character_system_messages = [
    generate_character_system_message(character_name, character_headers)
    for character_name, character_headers in zip(character_names, character_headers)
]
class BidOutputParser(RegexParser):
    def get_format_instructions(self) -> str:
        return "Your response should be an integer delimited by angled brackets, like this: <int>."


bid_parser = BidOutputParser(
    regex=r"<(\d+)>", output_keys=["bid"], default_output_key="bid"
)

@tenacity.retry(
    stop=tenacity.stop_after_attempt(2),
    wait=tenacity.wait_none(),  # No waiting time between retries
    retry=tenacity.retry_if_exception_type(ValueError),
    before_sleep=lambda retry_state: print(
        f"ValueError occurred: {retry_state.outcome.exception()}, retrying..."
    ),
    retry_error_callback=lambda retry_state: 0,
)  # Default value when all retries are exhausted

def ask_for_bid(agent) -> str:
    """
    Ask for agent bid and parses the bid into the correct format.
    """
    bid_string = agent.bid()
    bid = int(bid_parser.parse(bid_string)["bid"])
    return bid

def generate_character_bidding_template(character_header):
    bidding_template = f"""{character_header}

```
{{message_history}}
```

On the scale of 1 to 10, where 1 is least important to the startup pitch and 10 is extremely important and contribute, rank your recent message based on the context. Make sure to be very through in your ranking and only rank stuff that is important higher.

```
{{recent_message}}
```

{bid_parser.get_format_instructions()}
Do nothing else.
    """
    return bidding_template


character_bidding_templates = [
    generate_character_bidding_template(character_header)
    for character_header in character_headers
]


/tmp/ipython-input-3879484581.py:17: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  character_description = ChatOpenAI(temperature=0.6)(


### Define the speaker selection function
Lastly define a speaker selection function `select_next_speaker` that takes each agent's bid and selects the agent with the highest bid (with ties broken randomly).

Assume that you have a `ask_for_bid` function that takes in the agent and returns the numerical bid.

In [ ]:
import numpy as np


def select_next_speaker(step: int, agents: List[DialogueAgent]) -> int:
    bids = []
    for agent in agents:
        bid = ask_for_bid(agent)
        bids.append(bid)

    # randomly select among multiple agents with the same bid
    max_value = np.max(bids)
    max_indices = np.where(bids == max_value)[0]
    idx = np.random.choice(max_indices)

    print("Bids:")
    for i, (bid, agent) in enumerate(zip(bids, agents)):
        print(f"\t{agent.name} bid: {bid}")
        if i == idx:
            selected_name = agent.name
    print(f"Selected: {selected_name}")
    print("\n")
    return idx

### Creating Bidding Dialogue Agents for each Character
Assuming that for each character we have `character_name, character_system_message` and `bidding_template` write a loop that populates the characters list with the `BiddingDialogueAgent` objects for each character.


In [ ]:
characters = []
model=ChatOpenAI(temperature=0.4)


for character_name, character_system_message, bidding_template in zip(
    character_names, character_system_messages, character_bidding_templates
):
    characters.append(
        BiddingDialogueAgent(
            name=character_name,
            system_message=character_system_message,
            model=model,
            bidding_template=bidding_template,
        )
    )

### Run the simulation
Pulate the `first_message` field and also write the while loop to run the simulation.

In [ ]:
max_iters = 10
n = 0

simulator = DialogueSimulator(agents=characters, selection_function=select_next_speaker)
simulator.reset()

first_message = "CEO, CMO, CTO You can now start pitching your ideas to our investor Sandra and Daniel"
simulator.inject("Moderator", first_message )
print(f"(Moderator): {first_message}")
print("\n")

while n < max_iters:
    name, message = simulator.step()
    print(f"({name}): {message}")
    print("\n")
    n += 1

(Moderator): CEO, CMO, CTO You can now start pitching your ideas to our investor Sandra and Daniel


ValueError occurred: invalid literal for int() with base 10: '<int>9</int>', retrying...
Bids:
	CTO bid: 8
	CMO bid: 9
	CEO bid: 10
	Investor-Daniel bid: 8
	Investor-Sandra bid: 10
Selected: CEO


(CEO): *With a confident smile, I stand tall and begin my pitch with enthusiasm and energy.*


ValueError occurred: invalid literal for int() with base 10: '<int>9</int>', retrying...
ValueError occurred: invalid literal for int() with base 10: '<int>8</int>', retrying...
Bids:
	CTO bid: 8
	CMO bid: 7
	CEO bid: 0
	Investor-Daniel bid: 8
	Investor-Sandra bid: 8
Selected: CTO


(CTO): *Excitedly gesturing with my hands, I dive into our innovative caffeine-free energy drink concept.*


ValueError occurred: invalid literal for int() with base 10: '<int>9</int>', retrying...
ValueError occurred: invalid literal for int() with base 10: '<int>9</int>', retrying...
ValueError occurred: invalid literal

In [ ]:
from langchain.agents import AgentType, initialize_agent, load_tools


## `DialogueAgentWithTools` class
We define a `DialogueAgentWithTools` class that augments `DialogueAgent` to use tools.

In [ ]:
class DialogueAgentWithTools(DialogueAgent):
    def __init__(
        self,
        name: str,
        system_message: SystemMessage,
        model: ChatOpenAI,
        tool_names: List[str],
        **tool_kwargs,
    ) -> None:
        super().__init__(name, system_message, model)
        self.tools = load_tools(tool_names, **tool_kwargs)

    def send(self) -> str:
        """
        Applies the chatmodel to the message history
        and returns the message string
        """
        agent_chain = initialize_agent(
            self.tools,
            self.model,
            agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
            verbose=True,
            memory=ConversationBufferMemory(
                memory_key="chat_history", return_messages=True
            ),
        )
        message = AIMessage(
            content=agent_chain.run(
                input="\n".join(
                    [self.system_message.content] + self.message_history + [self.prefix]
                )
            )
        )

        return message.content

## Main Loop


In [ ]:
!pip install -U ddgs
# Agent Descriptions
agent_descriptions = {
    "Alex": "Alex is a strong advocate for remote work, emphasizing its flexibility and productivity benefits.",
    "Jordan": "Jordan is skeptical about remote work, focusing on potential downsides like reduced team interaction."
}


# System Messages
def generate_system_message(name, description):
    return f"""Your name is {name}.
#
          Your description is as follows: {description}

          Your goal is to persuade your conversation partner of your point of view.

          DO look up information with your tool to refute your partner's claims.
          DO cite your sources.

          DO NOT fabricate fake citations.
          DO NOT cite any source that you did not look up.

          Do not add anything else.

          Stop speaking the moment you finish speaking from your perspective.
          """

agent_system_messages = {name: generate_system_message(name, description) for name, description in agent_descriptions.items()}

# Topic Specification
specified_topic = "The Impact of Remote Work on Employee Productivity"

# Agent Setup
agents = [
    DialogueAgentWithTools(
        name=name,
        system_message=SystemMessage(content=system_message),
        model=ChatOpenAI(model_name="gpt-4", temperature=0.2),
        tool_names= ["arxiv", "ddg-search", "wikipedia"],
        top_k_results=2,
    ) for name, system_message in agent_system_messages.items()
]

# Speaker Selection Function
def select_next_speaker(step, agents):
    return step % len(agents)

# Running the Simulation
max_iters = 4
simulator = DialogueSimulator(agents=agents, selection_function=select_next_speaker)
simulator.reset()
simulator.inject("Moderator", specified_topic)

for _ in range(max_iters):
    name, message = simulator.step()
    print(f"({name}): {message}")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.3 MB/s eta 0:00:00


/tmp/ipython-input-2695530970.py:23: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(
/tmp/ipython-input-2695530970.py:18: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_chain = initialize_agent(
/tmp/ipython-input-2695530970.py:28: LangChainDeprecationWarning: The method `Chain.run` was



> Entering new AgentExecutor chain...
While many tout the benefits of remote work, I believe it's important to consider the potential downsides as well. One of the major concerns is the reduction in team interaction. While technology has made it easier for us to communicate, it's not a perfect substitute for face-to-face interaction. The spontaneous conversations that occur in an office setting can often lead to innovative ideas and solutions. These serendipitous interactions are hard to replicate in a remote work environment.

```json
{
    "action": "duckduckgo_search",
    "action_input": "impact of remote work on team interaction"
}
```
Observation: The shift to remote work has transformed the way teams interact , collaborate, and build relationships. 5. The Impact of Remote Work on Team Dynamics7. Best Practices for Enhancing Virtual Interactions In conclusion, the effectiveness of remote work relationships compared to in-person... Chapter 6: Discussion6.1 Impact of remote worki

>uper cell output
 Entering new AgentExecutor chain...
While many tout the benefits of remote work, I believe it's important to consider the potential downsides as well. One of the major concerns is the reduction in team interaction. While technology has made it easier for us to communicate, it's not a perfect substitute for face-to-face interaction. The spontaneous conversations that occur in an office setting can often lead to innovative ideas and solutions. These serendipitous interactions are hard to replicate in a remote work environment.

```json
{
    "action": "duckduckgo_search",
    "action_input": "impact of remote work on team interaction"
}
```
Observation: The shift to remote work has transformed the way teams interact , collaborate, and build relationships. 5. The Impact of Remote Work on Team Dynamics7. Best Practices for Enhancing Virtual Interactions In conclusion, the effectiveness of remote work relationships compared to in-person... Chapter 6: Discussion6.1 Impact of remote working to the effectiveness of team communication.6.2 The importance of face-to-face interaction . Conclusion: Remote work has altered team dynamics, posing both challenges and opportunities for managers. Adaptive management practices can help firms sustain team cohesion and productivity. Here the remote team acts as an extension of the team and both teams work towards the same goals. Transfer of ownership: The core product team intends to shift focus on the next generation of their product line and is unable to maintain/extend the existing product.
Thought:Based on the information I found, the shift to remote work has indeed transformed the way teams interact and collaborate. While it poses challenges, it also opens up new opportunities. The effectiveness of team communication in a remote setting can be comparable to in-person interaction, but it requires adaptive management practices. It's important to note that face-to-face interaction still holds its unique importance. Therefore, while remote work can be effective, it may require more deliberate efforts to maintain team cohesion and productivity.

```json
{
    "action": "Final Answer",
    "action_input": "Based on the information I found, the shift to remote work has indeed transformed the way teams interact and collaborate. While it poses challenges, it also opens up new opportunities. The effectiveness of team communication in a remote setting can be comparable to in-person interaction, but it requires adaptive management practices. It's important to note that face-to-face interaction still holds its unique importance. Therefore, while remote work can be effective, it may require more deliberate efforts to maintain team cohesion and productivity."
}
```

> Finished chain.
(Jordan): Based on the information I found, the shift to remote work has indeed transformed the way teams interact and collaborate. While it poses challenges, it also opens up new opportunities. The effectiveness of team communication in a remote setting can be comparable to in-person interaction, but it requires adaptive management practices. It's important to note that face-to-face interaction still holds its unique importance. Therefore, while remote work can be effective, it may require more deliberate efforts to maintain team cohesion and productivity.


> Entering new AgentExecutor chain...
```json
{
    "action": "duckduckgo_search",
    "action_input": "benefits of remote work on productivity"
}
```
Observation: May 22, 2025 - Of the companies on the 2025 Fortune 100 Best Companies to Work For®, 97 support remote or hybrid work. This study included 67,000 employees. And 84% of employees at these companies say they can count on colleagues to cooperate, compared to 65% in typical workplaces. Employees at the Fortune 100 Best Companies don’t just show up — they show up strong. Productivity is nearly 42% higher at these companies compared to a typical U.S. Aug 3, 2023 · One 2013 study from Stanford University examined a Chinese travel agency’s experience and concluded that remote work boosted performance and productivity by up to 22% over in-office work. Oct 31, 2024 · We find that TFP growth over both the 2019–21 and the 2019–22 periods is positively associated with the rise in the percentage of remote workers across 61 industries in the private business sector, even after accounting for pre-pandemic trends in productivity . Currently, remote work has become a crucial organizational tool that enables effective performance in the increasingly competitive global market. Although working outside of the office has already been available, this form of performing job duties seems mainstream in modern organizations. A study conducted by Stanford University in the summer of 2020 discovered that remote workers were 5 percent more productive than those working in a physical office. By the spring of 2022, remote worker productivity had risen to 9 percent as businesses became more familiar with remote work practices and invested in technology to support it. This tr... See full list on psychologytoday.com Similarly, a large financial services company reported that remote work led to improved focus and concentration among their employees. They found that employees were better able to stay on task and avoid distractions when working from home. This led to a significant increase in productivity and helped the company to achieve its goals. Furthermore, ... See full list on psychologytoday.com Another benefit of remote work is that it leads to improved work –life balance, which prevents burnoutand, thus, increases productivity and retention alike. A midsize retail company found that remote work allowed their back-office employees to better balance their work and personal responsibilities. This led to happier and more fulfilled employees, ... See full list on psychologytoday.com To fully realize the benefits of remote work , businesses need to ensure that their employees are equipped with the right tools and resources. This includes providing employees with the necessary technology to work remotely, such as laptops, Internet connectivity, and collaborationtools. Businesses also need to provide their employees with the neces... See full list on psychologytoday.com Remote work productivity is a key to success in the modern business world. The benefits of remote work, including increased productivity, improved focus and concentration, and improved work–life balance , are too significant to be ignored. Business leaders who want to stay competitive and succeed in the long term should seriously consider implementi... See full list on psychologytoday.com Jun 12, 2024 · In the largest study yet of working -from-home professionals, Stanford economist Nicholas Bloom reveals that employees who work from home two days a week are just as productive , likely to get promoted, and far less prone to quit. Jan 17, 2025 · Remote work has transformed the modern workplace by offering flexibility and reducing commute times for employees. While evidence suggests it has generally resulted in higher productivity ,...
Thought:```json
{
    "action": "Final Answer",
    "action_input": "Indeed, Jordan, while remote work does require adaptive management practices, it has been shown to significantly boost productivity. For instance, a study from Stanford University found that remote work increased performance by up to 22% compared to in-office work. Moreover, productivity was nearly 42% higher at companies that support remote or hybrid work, according to a survey of the 2025 Fortune 100 Best Companies to Work For®. Another study conducted by Stanford University in 2020 found that remote workers were 5% more productive than their office-based counterparts, and this figure rose to 9% by the spring of 2022 as businesses became more adept at implementing remote work practices. Furthermore, remote work has been found to improve focus and concentration, leading to increased productivity. It also promotes a better work-life balance, which can prevent burnout and increase both productivity and retention. Therefore, it's clear that remote work, when properly implemented, can have substantial benefits for both employees and organizations."
}
```

> Finished chain.
(Alex): Indeed, Jordan, while remote work does require adaptive management practices, it has been shown to significantly boost productivity. For instance, a study from Stanford University found that remote work increased performance by up to 22% compared to in-office work. Moreover, productivity was nearly 42% higher at companies that support remote or hybrid work, according to a survey of the 2025 Fortune 100 Best Companies to Work For®. Another study conducted by Stanford University in 2020 found that remote workers were 5% more productive than their office-based counterparts, and this figure rose to 9% by the spring of 2022 as businesses became more adept at implementing remote work practices. Furthermore, remote work has been found to improve focus and concentration, leading to increased productivity. It also promotes a better work-life balance, which can prevent burnout and increase both productivity and retention. Therefore, it's clear that remote work, when properly implemented, can have substantial benefits for both employees and organizations.


> Entering new AgentExecutor chain...
```json
{
    "action": "duckduckgo_search",
    "action_input": "downsides of remote work"
}
```
Observation: May 21, 2025 - Remote work offers higher productivity and access to global talent, but also brings challenges like isolation and communication gaps . August 28, 2025 - Quite a few companies may even fail to provide company laptops to their staff. Forgetting their password or being locked out of their computer or account are just a few things that can happen while working remotely. June 11, 2025 - Employees can’t just run out for a personal errand or take an hour-long nap break because they aren’t able to focus on work. They can only do all that after work hours or by taking an official leave from work. On the contrary, remote employees, especially those enjoying flexible work, get more freedom to create their own schedule. July 20, 2025 - Team communication barriers : Lack of in-person interaction can cause miscommunication, weakened collaboration, and slower decision-making. Technological hurdles: Internet connectivity issues and over-reliance on digital tools can disrupt workflows. March 26, 2025 - Many find themselves working longer hours than they might in a traditional office because there’s no physical separation to signal the end of the workday. And while wearing pyjamas during a meeting has its perks, the lack of direct, face-to-face interaction can leave you feeling isolated and a bit out of the loop. So, what exactly is behind these growing disadvantages of remote work for employees and—more importantly—what’s the solution?
Thought:```json
{
    "action": "Final Answer",
    "action_input": "Alex, while the studies you mentioned do highlight the potential productivity gains from remote work, it's important to consider the other side of the coin. For instance, remote work can lead to challenges such as isolation and communication gaps. The lack of in-person interaction can cause miscommunication, weakened collaboration, and slower decision-making. Technological hurdles like internet connectivity issues and over-reliance on digital tools can disrupt workflows. Additionally, many remote workers find themselves working longer hours than they might in a traditional office because there’s no physical separation to signal the end of the workday. This blurring of boundaries between work and personal life can lead to burnout, negating the productivity gains. Therefore, while remote work has its advantages, it's crucial to address these challenges to ensure its long-term sustainability and effectiveness."
}
```

> Finished chain.
(Jordan): Alex, while the studies you mentioned do highlight the potential productivity gains from remote work, it's important to consider the other side of the coin. For instance, remote work can lead to challenges such as isolation and communication gaps. The lack of in-person interaction can cause miscommunication, weakened collaboration, and slower decision-making. Technological hurdles like internet connectivity issues and over-reliance on digital tools can disrupt workflows. Additionally, many remote workers find themselves working longer hours than they might in a traditional office because there’s no physical separation to signal the end of the workday. This blurring of boundaries between work and personal life can lead to burnout, negating the productivity gains. Therefore, while remote work has its advantages, it's crucial to address these challenges to ensure its long-term sustainability and effectiveness.


> Entering new AgentExecutor chain...
```json
{
    "action": "duckduckgo_search",
    "action_input": "benefits of remote work overcoming challenges"
}
```
Observation: Benefits and challenges of remote workers . Lucid Content. Reading time: about 9 min.Global Workplace Analytics, remote work as grown by 173% since 2005. Obviously, there is a strong case for companies to better accommodate remote workers . Overcoming Remote Work Challenges . Despite the clear benefits , remote work also presents challenges . Employees may experience isolation due to a lack of in-person interaction, and it requires self-discipline to remain productive. Overcoming Remote Work Challenges . To overcome these challenges , companies can implement several strategies.While there are challenges to address, the benefits of remote work are undeniable. High productivity isn’t the only universal benefit of working remotely .While these challenges are a reality, proactively addressing them is the first step toward overcoming remote work challenges and ensuring a successful work -from-home career. 9. Generational support of flexible work . Possible Benefits and Challenges for Remote Work . Overcoming the Challenges *. Decision to Approve/Deny a Request to Work Remotely and Telecommuting Agreement. References. Considerations for Partial or Fully Remote Work Supervisor Guide.
Thought:```json
{
    "action": "Final Answer",
    "action_input": "Jordan, you've raised valid points about the challenges of remote work, such as isolation, communication gaps, and the blurring of work-life boundaries. However, it's important to note that these challenges can be effectively managed with the right strategies. For instance, companies can foster a sense of community among remote workers through regular virtual meetings and team-building activities. They can also provide clear communication guidelines and invest in reliable technology to minimize communication gaps and technical issues. As for the blurring of work-life boundaries, companies can encourage employees to establish a regular work schedule and create a dedicated workspace at home. This can help maintain a clear distinction between work and personal life, preventing burnout and promoting productivity. Therefore, while remote work does present certain challenges, these can be overcome with proactive measures, allowing companies to reap the benefits of increased productivity, flexibility, and employee satisfaction. This perspective is supported by a Lucid Content article that discusses the benefits and challenges of remote work and how to overcome them."
}
```

> Finished chain.
(Alex): Jordan, you've raised valid points about the challenges of remote work, such as isolation, communication gaps, and the blurring of work-life boundaries. However, it's important to note that these challenges can be effectively managed with the right strategies. For instance, companies can foster a sense of community among remote workers through regular virtual meetings and team-building activities. They can also provide clear communication guidelines and invest in reliable technology to minimize communication gaps and technical issues. As for the blurring of work-life boundaries, companies can encourage employees to establish a regular work schedule and create a dedicated workspace at home. This can help maintain a clear distinction between work and personal life, preventing burnout and promoting productivity. Therefore, while remote work does present certain challenges, these can be overcome with proactive measures, allowing companies to reap the benefits of increased productivity, flexibility, and employee satisfaction. This perspective is supported by a Lucid Content article that discusses the benefits and challenges of remote work and how to overcome them.